In [15]:
import numpy as np

# STEP 1: Define the Hidden States (Tags)

tags = ["Noun", "Verb", "Adjective"]


# STEP 2: Define the Vocabulary

words = ["Afreen", "is", "beautifull"]

# Convert words into indices
word_to_index = {
    "Afreen": 0,
    "is": 1,
    "beautifull": 2
}

# STEP 3: Initial Probability
# P(Tag at first word)

init_prob = np.array([
    0.7,  # Noun
    0.2,  # Verb
    0.1   # Adjective
])

# STEP 4: Transition Probability Matrix
# P(Current Tag | Previous Tag)
#
#              To
#         Noun  Verb  Adjective
# From Noun
#      Verb
#      Adjective

transition = np.array([
    [0.1, 0.8, 0.1],   # From Noun
    [0.1, 0.1, 0.8],   # From Verb
    [0.34, 0.33, 0.33] # From Adjective
])


# STEP 5: Emission Probability Matrix
# P(Word | Tag)
#
#            Afreen  is    beautyfull
# Noun        0.8    0.1   0.1
# Verb        0.1    0.8   0.1
# Adjective   0.1    0.1   0.8

emission = np.array([
    [0.8, 0.1, 0.1],
    [0.1, 0.8, 0.1],
    [0.1, 0.1, 0.8]
])


# STEP 6: Convert probabilities into Log Space
# This avoids numerical underflow.

log_init = np.log(init_prob + 1e-10)
log_trans = np.log(transition + 1e-10)
log_emit = np.log(emission + 1e-10)

# STEP 7: Viterbi Algorithm

def viterbi(sentence):
    # Number of words
    T = len(sentence)
    # Number of tags
    N = len(tags)

    # DP table
    dp = np.full((N, T), -np.inf)

    # Backpointer table
    backpointer = np.zeros((N, T), dtype=int)


    # Calculate probability for the first word
    dp[:, 0] = log_init + log_emit[:, sentence[0]]


    # Process remaining words
    for t in range(1, T):
        # Check every current tag
        for j in range(N):
            # Calculate score from every previous tag
            scores = (
                dp[:, t-1]
                + log_trans[:, j]
                + log_emit[j, sentence[t]]
            )
            # Store best previous tag
            backpointer[j, t] = np.argmax(scores)
            # Store best score
            dp[j, t] = np.max(scores)

    
    # Best tag for last word
    best_path = [np.argmax(dp[:, -1])]

    # Trace backwards
    for t in range(T-1, 0, -1):
        best_path.append(backpointer[best_path[-1], t])

    # Reverse to get correct order
    best_path.reverse()

    return best_path

# STEP 8: Example Sentence

sentence = ["Afreen", "is", "beautifull"]

# Convert words into indices
sentence_index = [word_to_index[word] for word in sentence]

# Run Viterbi
best_tags = viterbi(sentence_index)


# STEP 9: Print Results

print("Sentence:", sentence)
print("\nPredicted Tags:")
for word, tag in zip(sentence, best_tags):
    print(word, "->", tags[tag])

Sentence: ['Afreen', 'is', 'beautifull']

Predicted Tags:
Afreen -> Noun
is -> Verb
beautifull -> Adjective


In [ ]:
#-----------------Summary-----------------------
#   POS tagging assigns grammatical labels (noun, verb, adjective, etc.) to words.
#   HMM predicts the hidden sequence of POS tags using:
#   Transition probability: how likely one tag follows another.
#   Emission probability: how likely a tag generates a particular word.
#   Checking all possible tag sequences is computationally infeasible because the number of combinations grows exponentially.
#   The Viterbi algorithm uses dynamic programming to efficiently find the most probable tag sequence.
#   Log probabilities are used to avoid numerical underflow when combining many small probability values